In [5]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

In [6]:
docs = [
    Document(page_content="Langchain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search"),
    Document(page_content="The eiffel tower is located in paris"),
    Document(page_content="Langchain can be used to develop agentic ai application"),
    Document(page_content="Langchain has many types of retrievers")
]
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
dense_vectorstore = FAISS.from_documents(
    docs,embedding_model
)
dense_retriver = dense_vectorstore.as_retriever()

/tmp/ipykernel_4165/3242597675.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
# Sparse retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3 # Top k documents

# Combine with ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers = [dense_retriver,sparse_retriever],
    weight = [0.7,0.3] 
)
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f540304cb00>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f53fe2bf0b0>, k=3)], weights=[0.5, 0.5])

In [8]:
query = "How can i build an application using LLMS?"

results = hybrid_retriever.invoke(query)

for i,doc in enumerate(results):
    print(f"\n Document{i+1}:\n{doc.page_content}")


 Document1:
Langchain helps build LLM applications.

 Document2:
Langchain can be used to develop agentic ai application

 Document3:
Langchain has many types of retrievers

 Document4:
Pinecone is a vector database for semantic search


In [18]:
# Rag pipeline
import os
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [19]:
llm = ChatGroq(
     model="openai/gpt-oss-20b",
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )
prompt = PromptTemplate.from_template("""
    Answer the question based on the context below.
    
    context:{context}
    Question:{input}"""
)

In [21]:
document_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(hybrid_retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f540304cb00>, search_kwargs={}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x7f53fe2bf0b0>, k=3)], weights=[0.5, 0.5]), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the question based on the context below.\n\n    context:{context}\n    Question:{input}')
            | ChatGroq(metadata={'l

In [32]:
query = {"input":"How can i build app using LLMS?"}
response = rag_chain.invoke(query)
print("Answer:\n", response["answer"][:500])

print("\nSource Documents:")
for i,doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}")

Answer:
 Below is a practical, step‑by‑step guide to building a simple **LLM‑powered app** that uses LangChain for orchestration and Pinecone for semantic search.  
Feel free to adapt the code snippets to your own language, framework, or deployment stack.

---

## 1. Choose the core components

| Component | What it does | Typical choice |
|-----------|--------------|----------------|
| **LLM** | Generates text, answers questions, writes code, etc. | OpenAI GPT‑4, Anthropic Claude, Cohere, Llama‑2, etc. 

Source Documents:

Doc 1: Langchain helps build LLM applications.

Doc 2: Langchain can be used to develop agentic ai application

Doc 3: Langchain has many types of retrievers

Doc 4: Pinecone is a vector database for semantic search
